# Get to Know a Dataset: GeoNadir Fair Geo

This notebook is a guided tour of the [GeoNadir Fair Geo](https://registry.opendata.aws/geonadir-fair-geo) collection: open UAV (drone) survey datasets contributed through GeoNadir and published as cloud-native geospatial files. You can also browse the collection in your browser at [https://data.geonadir.com/fairgeo](https://data.geonadir.com/fairgeo). More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

> **This is the STAC version.** It discovers datasets through the GeoNadir Fair Geo STAC API. A companion notebook, `get-to-know-a-dataset.ipynb`, does the same tour using **plain AWS S3 only** (no STAC) — useful before the STAC endpoint is live, or if you prefer not to depend on a STAC client.

> **Draft status.** Cells that only need the public S3 bucket are marked **[RUNS AT LAUNCH]**. Cells that need the STAC API online are marked **[NEEDS STAC]**.

## Understanding the dataset structure

GeoNadir Fair Geo is organized so the files are simple to address directly. Each UAV survey is one folder under the `datasets/` prefix, keyed by a stable public item id:

```text
s3://geonadir-fair-geo/
  stac/                         <- STAC catalog (optional discovery layer)
  datasets/
    {item_id}/
      metadata.json             (required)
      raw-images.zip            (required)
      orthomosaic.tif           (optional, Cloud Optimized GeoTIFF)
      dsm.tif                   (optional, Cloud Optimized GeoTIFF)
      dtm.tif                   (optional, Cloud Optimized GeoTIFF)
      multispectral-orthomosaic.tif  (optional, Cloud Optimized GeoTIFF)
```

One UAV survey = one `datasets/{item_id}/` folder. Only the files that exist for a survey are present, so a survey with no DTM simply has no `dtm.tif`.

The STAC catalog under `stac/` is the discovery layer used in this notebook — you search it and it hands back direct links to these files, so you never have to browse the bucket by hand.

In [ ]:
# CODING GUIDELINES (from the AWS template):
# This is a 101-level introduction using common tools and libraries.
# The bucket is public, so all S3 reads are UNSIGNED (no AWS account or credentials needed).

First, the libraries used throughout this notebook. The dataset is public, so S3 reads are unsigned.

In [ ]:
# This notebook requires the following additional libraries
# (install with pip or conda for your environment):
#
# boto3 >= 1.34
# rasterio >= 1.3
# pystac-client >= 0.7
# matplotlib >= 3.8
# numpy >= 1.26

# Built-ins
import json
from pprint import pprint

# Installed libraries
import boto3
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import rasterio.session
from botocore import UNSIGNED
from botocore.config import Config
from pystac_client import Client

Next, the dataset location. Update these if the final published names differ from the draft.

In [ ]:
# [RUNS AT LAUNCH] Dataset location
BUCKET = "geonadir-fair-geo"
REGION = "ap-southeast-2"
STAC_API = "https://data.geonadir.com/stac"
COLLECTION = "geonadir-fair-geo"

# Public bucket -> unsigned S3 requests (no credentials required)
s3 = boto3.client('s3', region_name=REGION, config=Config(signature_version=UNSIGNED))
aws_session = rasterio.session.AWSSession(aws_unsigned=True)  # for reading COGs from s3://

With STAC we can ask for exactly the surveys we want — by area of interest, date range, and habitat — instead of listing the bucket.

In [ ]:
# [NEEDS STAC] Open the catalog and search
catalog = Client.open(STAC_API)
search = catalog.search(
    collections=[COLLECTION],
    datetime="2024-01-01T00:00:00Z/2024-12-31T23:59:59Z",
    intersects={
        "type": "Polygon",
        "coordinates": [[[151.0, -34.0], [151.2, -34.0], [151.2, -33.8], [151.0, -33.8], [151.0, -34.0]]],
    },
    filter="geonadir:iucn_habitat = 'Forest'",
    max_items=10,
)
items = list(search.items())
print(f"Found {len(items)} matching survey(s)")
for it in items:
    print(it.id, it.datetime, sorted(it.assets.keys()))

## Data formats

GeoNadir Fair Geo uses a small, deliberately standard set of formats so the data works with off-the-shelf tools:

| Asset | Format | Why this format |
| --- | --- | --- |
| `metadata.json` | JSON | Normalized survey metadata; readable in any language |
| `raw-images.zip` | ZIP of JPEG | Original UAV frames kept together for reprocessing / photogrammetry |
| `orthomosaic.tif`, `dsm.tif`, `dtm.tif`, `multispectral-orthomosaic.tif` | Cloud Optimized GeoTIFF (COG) | Read a window or overview **directly from S3** without downloading the whole file |

The COG is the key choice. A Cloud Optimized GeoTIFF is a normal GeoTIFF with internal tiling + overviews arranged so an HTTP range-request pulls only the pixels you ask for. GDAL, Rasterio, QGIS, and cloud-native pipelines can open a multi-gigabyte orthomosaic and read a small window or a low-res overview in seconds. Each COG carries its own CRS and geotransform, so projection metadata is read straight from the file. Bands follow the common EO convention (`B1`, `B2`, ... with `common_name` like `red`, `green`, `blue`, `rededge`, `nir`).

## Loading data from a survey

First the small `metadata.json` for one survey, then — the interesting part — opening an orthomosaic COG and reading only a low-resolution overview, straight from S3, without downloading the full GeoTIFF.

In [ ]:
# [NEEDS STAC] Pick the first item from the search above
item = items[0]
item_id = item.id

# Its metadata.json href comes straight off the STAC 'metadata' asset
meta_href = item.assets['metadata'].href  # s3://.../datasets/{item_id}/metadata.json
key = meta_href.split(f'{BUCKET}/', 1)[1]
with s3.get_object(Bucket=BUCKET, Key=key)['Body'] as body:
    metadata = json.load(body)
pprint(metadata)

Now the orthomosaic. We open it with Rasterio over `s3://` (GDAL's `/vsis3/`). Because it is a COG, reading a decimated overview transfers only a few tiles — not the whole file.

In [ ]:
# [NEEDS STAC] Resolve the orthomosaic href from the STAC item
ortho_href = item.assets['orthomosaic'].href

with rasterio.Env(session=aws_session):
    with rasterio.open(ortho_href) as src:
        print('CRS:', src.crs)
        print('Size (WxH):', src.width, 'x', src.height)
        print('Bands:', src.count)
        print('Resolution (m):', src.res)
        print('Bounds:', src.bounds)
        # Read a downsampled overview: at most ~1024 px on the long edge
        scale = max(1, max(src.width, src.height) // 1024)
        out_shape = (src.count, src.height // scale, src.width // scale)
        overview = src.read(out_shape=out_shape)
print('Overview array shape (bands, rows, cols):', overview.shape)

## Visualizing a survey

Two visuals. First, the RGB orthomosaic overview we just read — the survey as an image. Second, a collection-level view: where the surveys are, colored by IUCN habitat.

In [ ]:
# [RUNS AT LAUNCH] Display the RGB orthomosaic overview
rgb = np.transpose(overview[:3], (1, 2, 0)).astype('float32')  # first 3 bands: R,G,B
rgb = (rgb - rgb.min()) / (rgb.ptp() + 1e-9)                   # normalize to 0-1 for display

plt.figure(figsize=(9, 9), dpi=100)
plt.imshow(rgb)
plt.title(f'GeoNadir Fair Geo orthomosaic overview: {item_id}', fontsize=13, fontweight='bold')
plt.axis('off')
plt.show()

For the collection footprint, we gather each survey's bounding-box centroid and habitat, then scatter them on a simple map — the kind of overview that tells a new user what the collection actually covers.

In [ ]:
# [NEEDS STAC] Map every published survey by centroid, colored by IUCN habitat
all_items = list(catalog.search(collections=[COLLECTION]).items())
records = [((it.bbox[0]+it.bbox[2])/2, (it.bbox[1]+it.bbox[3])/2,
            it.properties.get('geonadir:iucn_habitat', 'Unknown')) for it in all_items]

In [ ]:
# [RUNS AT LAUNCH] Plot the centroids
lons = [r[0] for r in records]; lats = [r[1] for r in records]; habitats = [r[2] for r in records]
cats = sorted(set(habitats))
cmap = {c: plt.cm.tab20(i % 20) for i, c in enumerate(cats)}
plt.figure(figsize=(12, 7), dpi=100)
for c in cats:
    xs = [lo for lo, h in zip(lons, habitats) if h == c]
    ys = [la for la, h in zip(lats, habitats) if h == c]
    plt.scatter(xs, ys, label=c, s=40, alpha=0.8, color=cmap[c])
plt.title('GeoNadir Fair Geo — published surveys by IUCN habitat', fontsize=14, fontweight='bold')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.grid(True, linestyle='--', alpha=0.3); plt.tight_layout(); plt.show()

## Worked example: measuring vegetation cover

**How much of a surveyed site is vegetated?** For a multispectral survey we compute NDVI (Normalized Difference Vegetation Index) directly from the red and near-infrared bands of the multispectral orthomosaic, then report the vegetated fraction of the survey. A common first-pass measure for habitat and land-cover work — and it runs off the COG without downloading the full raster.

In [ ]:
# [NEEDS STAC] Compute NDVI from the multispectral orthomosaic and report vegetated fraction.
# Band order for GeoNadir multispectral COGs (see OPEN_DATA_STRUCTURE.md): B1 green, B2 red, B3 rededge, B4 nir.
ms_href = item.assets['multispectral_orthomosaic'].href
with rasterio.Env(session=aws_session):
    with rasterio.open(ms_href) as src:
        scale = max(1, max(src.width, src.height) // 2048)
        out_shape = (src.count, src.height // scale, src.width // scale)
        arr = src.read(out_shape=out_shape).astype('float32')

red = arr[1]  # B2
nir = arr[3]  # B4
ndvi = (nir - red) / (nir + red + 1e-9)
valid = np.isfinite(ndvi) & (arr.sum(axis=0) > 0)
vegetated = ndvi > 0.3            # common threshold for healthy vegetation
frac = vegetated[valid].mean() if valid.any() else float('nan')
print(f'Vegetated fraction of survey (NDVI > 0.3): {frac:.1%}')

plt.figure(figsize=(9, 8), dpi=100)
plt.imshow(np.where(valid, ndvi, np.nan), cmap='RdYlGn', vmin=-0.2, vmax=0.9)
plt.colorbar(label='NDVI')
plt.title(f'NDVI — {item_id}', fontsize=13, fontweight='bold')
plt.axis('off'); plt.show()

## Community challenge

**Habitat-scale change and classification from citizen UAV surveys.**

GeoNadir Fair Geo is a growing, globally distributed collection of centimetre-scale UAV surveys tagged with IUCN habitat classes. That combination is unusual: open satellite data is plentiful but coarse; centimetre-scale drone imagery with open licensing and consistent metadata is rare. Some questions we would love the community to take on:

1. **Habitat classification transfer.** Train a land-cover / habitat classifier on the multispectral + RGB orthomosaics and test how well it generalizes across IUCN habitat classes and geographies.
2. **Super-resolution priors for satellite data.** Use the centimetre-scale orthomosaics as ground truth to train or evaluate super-resolution / downscaling of Sentinel-2 or Landsat over the same footprints.
3. **Structure from the DSM/DTM pair.** Where both exist, derive canopy height models and study vegetation structure against the recorded habitat class.

**Getting started.** Filter the STAC catalog by `geonadir:iucn_habitat` to assemble a per-habitat training set. Read overviews from the COGs first (fast, cheap) before committing to full-resolution reads, and treat `geonadir:captured_by` / `geonadir:institution` as uploader-supplied attribution rather than verified authority records. Start with a toy subset of a handful of surveys — a working small example beats a stalled large one.

---
*Contributions and derived tutorials are welcome — open a pull request against [open-data-examples](https://github.com/Geonadir/open-data-examples) or add your tutorial to the GeoNadir Fair Geo entry on the Registry of Open Data.*